In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import sys
import numpy as np
import shutil
import pandas as pd
import itertools
import json
import os
import time
import torch
from IPython.display import clear_output, display, Image
from transformers import AutoTokenizer
sys.path.append('/Users/orenm/Desktop/code_projects/BlenderShaderProject/project_files/')

In [2]:
from Logic.tree_networks_manager import TreesNetworkManager
from Logic.mcts_operator import MCTSOperator, search_metrics
from Logic.utils import show_image_grid
from Logic.training_scripts.image_to_code_training import ImageCodeDataset
from Logic.NN_models.images_to_code_model import ImageToCodeDecoder, SpecialTokenTokenizerWrapper
from Logic.NN_models.image_embedders import load_resnet_model
from Logic.blender_tree_manager import BlenderTreeManager
from Logic.bpy_connector import generate_image
from Logic.variations_creator import apply_variation

In [3]:
path = '/Users/orenm/Desktop/code_projects/BlenderShaderProject/data/'
active_models_path = os.path.join(path, "active_models/")
mcts_workdir = os.path.join(path, 'mcts_work_dir')
comparison_experiment_dir = os.path.join(path, 'comparison_experiment')
real_textures = os.path.join(comparison_experiment_dir, 'real_textures_test_images')
held_out_textures = os.path.join(comparison_experiment_dir, 'generated_test_images')
results_dir = os.path.join(comparison_experiment_dir, 'results')

In [4]:
correction_model = 'balanced_models_ep_2_code_corrector_be_small_ru_3_we_0_001_le_5e-06.pt'
image_embedder_for_texture = 'ep_4_la_7_256_le_0_0001_mo_resnet_fi_128_sc_cosine.pt'
code_emb_file_name = 'ep_12_code_emb_be_mine_we_0_001_le_1e-05_be_big_3.pt'
image_emb_file_name = code_emb_file_name.replace('code_emb', 'image_emb')
correction_model_path = os.path.join(active_models_path, correction_model)
image_embedder_for_texture_path = os.path.join(active_models_path, image_embedder_for_texture)
tokenizer_path = os.path.join(active_models_path, "my_tokenizer")
iterations_dir =  os.path.join(comparison_experiment_dir, 'search_iterations')
subgraphs_dir = os.path.join(comparison_experiment_dir, 'example_networks')
mcts_operator = MCTSOperator(correction_model_path, tokenizer_path, image_embedder_for_texture_path)

In [5]:
def copy_selected_pngs(src_folder, dst_parent_folder, new_folder_name, filenames):
    """
    filenames: list of PNG filenames, e.g. ["a.png", "b.png"]
    """
    dst_folder = os.path.join(dst_parent_folder, new_folder_name)
    os.makedirs(dst_folder, exist_ok=True)

    for name in filenames:
        src_path = os.path.join(src_folder, name)
        dst_path = os.path.join(dst_folder, name)

        if os.path.isfile(src_path) and name.lower().endswith(".png"):
            shutil.copy2(src_path, dst_path)
    return dst_folder

def json_safe(obj):
    if isinstance(obj, np.generic):  # np.float32, np.int64, etc.
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Type {type(obj)} not serializable")

In [6]:
all_paths = []
for file_name in os.listdir(real_textures):
    all_paths.append((os.path.join(real_textures, file_name), 'new_textures'))
for file_name in os.listdir(held_out_textures):
    all_paths.append((os.path.join(held_out_textures, file_name), 'held_out'))

In [7]:
search_methods = ["mcts", "sample"]

# with tree search

In [9]:
with_iteration_trees = False
max_expansions = 30
max_nodes_to_expand_per_iter = 12

t = time.time()
for i, (file_path, test_set) in enumerate(all_paths):
    for search_method in search_methods:
        print(test_set, f'{i}/{len(all_paths)}', search_method, False, time.time()-t)
        target_id = file_path.split('\\')[-1].replace('.png', '')
        run_name = f'{target_id}_search_{search_method}_{max_expansions}_{max_nodes_to_expand_per_iter}'
        if run_name + ".json" in os.listdir(results_dir):
            continue

        for i in range(3):  # sometimes there are invalid multinomial distribution errors
            try:
                graph_manager, all_node_expansions, times = mcts_operator.search(
                    file_path,
                    mcts_workdir,
                    max_expansions=max_expansions,
                    max_nodes_to_expand_per_iter=max_nodes_to_expand_per_iter,
                    search_method=search_method,
                    c_puct=3.3,
                    sample_labels=True,
                    optimize=False,
                    n_nodes_to_optimize_at_end=3,
                    return_time_stats = True
                )
                break
            except:
                continue
        clear_output()

        # copy all iterations images
        all_visited_nodes = [node_name for expansion in all_node_expansions for node_name in expansion]
        all_new_images = [node_name+'.png' for node_name in all_visited_nodes]
        run_img_path = copy_selected_pngs(mcts_workdir, iterations_dir, run_name , all_new_images)

        # get all iteration scores
        best_so_far = -1
        iterations_log = []
        for iteration_num, nodes in enumerate(all_node_expansions):
            similarity_scores = [(graph_manager.network.nodes[node_name]['texture_similarity_value'], node_name) for node_name in nodes]
            score, best_node = max(similarity_scores)
            best_so_far = max(score, best_so_far)
            best_node_tree = None
            if with_iteration_trees:
                best_node_tree = graph_manager.blender_tree_managers[best_node].to_str()
            iterations_log.append({
                'iteration_num': iteration_num,
                'best_so_far_score': best_so_far,
                'best_in_iteration_score': score,
                'best_in_iteration_image_path': os.path.join(run_img_path, best_node+'.png'),
                'num_nodes_in_iteration': len(nodes),
                'best_node_tree': best_node_tree,
            })

        # get best overall
        similarity_scores = [
            (graph_manager.network.nodes[node_name]['texture_similarity_value'], node_name) for node_name in all_visited_nodes
        ]
        score, best_node = max(similarity_scores)
        best_tree_text = graph_manager.blender_tree_managers[best_node].to_str()

        subgraph_representation, nodes_included = graph_manager.create_partial_str_representation(best_node, 4)
        subgraph_images = [node_name+'.png' for node_name in nodes_included]
        dst_folder = copy_selected_pngs(mcts_workdir, subgraphs_dir, run_name, subgraph_images)
        open(os.path.join(dst_folder, "subgraph.txt"), "w", encoding="utf-8").write(subgraph_representation)

        res = {
            'target_id': target_id,
            'run_name': run_name,
            'target_path': file_path,
            'method': search_method,
            'optimization': False,
            'test_set': test_set,
            'best_similarity': score,
            'best_image_path': os.path.join(run_img_path, best_node+'.png'),
            'inference_times': times,
            'num_iterations': max_expansions,
            'max_nodes_to_expand_per_iter': max_nodes_to_expand_per_iter,
            'best_tree_text': best_tree_text,
            'iterations_log': iterations_log,
        }
        with open(os.path.join(results_dir, run_name + ".json"), "w") as f:
            json.dump(res, f, indent=4, default=json_safe)

new_textures 0/233 mcts False 0.0
new_textures 0/233 sample False 0.004188060760498047
new_textures 1/233 mcts False 0.004188060760498047
new_textures 1/233 sample False 0.010338544845581055
new_textures 2/233 mcts False 0.012345552444458008
new_textures 2/233 sample False 0.01435089111328125
new_textures 3/233 mcts False 0.01435089111328125
new_textures 3/233 sample False 0.01435089111328125
new_textures 4/233 mcts False 0.019843578338623047
new_textures 4/233 sample False 0.021837234497070312
new_textures 5/233 mcts False 0.023897171020507812
new_textures 5/233 sample False 0.023897171020507812
new_textures 6/233 mcts False 0.023897171020507812
new_textures 6/233 sample False 0.023897171020507812
new_textures 7/233 mcts False 0.023897171020507812
new_textures 7/233 sample False 0.023897171020507812
new_textures 8/233 mcts False 0.023897171020507812
new_textures 8/233 sample False 0.0386655330657959
new_textures 9/233 mcts False 0.041844844818115234
new_textures 9/233 sample False 0.0

# decoder comparison

In [9]:
IMAGE_EMBEDDER_OUTPUT_DIM = 128
codes_file = os.path.join(path, 'datasets/network_managers_strings.json')
image_embedder_for_texture = 'ep_4_la_7_256_le_0_0001_mo_resnet_fi_128_sc_cosine.pt'
image_embedder_for_texture_path = os.path.join(active_models_path, image_embedder_for_texture)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
tokenizer = SpecialTokenTokenizerWrapper(tokenizer)
image_embedder = load_resnet_model(image_embedder_for_texture_path)
MODEL_SAVE_PATH = os.path.join(active_models_path, "image_to_code_decoder_final.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
strings = json.load(open(codes_file, "r"))
lengths = [len(tokenizer.encode(x)) for x in strings.values()]
MAX_SEQUENCE_LENGTH = max(lengths) + 15

In [10]:
loaded_model = ImageToCodeDecoder(
    embedder=image_embedder,
    tokenizer=tokenizer,
    vocab_size=tokenizer.vocab_size,
    image_emb_dim=IMAGE_EMBEDDER_OUTPUT_DIM,
    model_dim=768,
    num_layers=16,
    num_heads=12,
    max_seq_len=MAX_SEQUENCE_LENGTH,
    pad_token_id=tokenizer.pad_token_id,
    sos_token_id=tokenizer.sos_token_id,
    eos_token_id=tokenizer.eos_token_id
).to(device)
loaded_model.load_state_dict_from_file(MODEL_SAVE_PATH, device=device)
loaded_model.eval()  # Set to evaluation mode for inference

Model state_dict loaded from /Users/orenm/Desktop/code_projects/BlenderShaderProject/data/active_models/image_to_code_decoder_final.pth


ImageToCodeDecoder(
  (embedder): Sequential(
    (0): Sequential(
      (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, 

In [15]:
num_solutions_per_try_options = [7, 2, 20]

res = []
dataset = ImageCodeDataset(image_dir='',codes_filepath=codes_file,tokenizer=None,)  # only used to open the image, not very efficient
t = time.time()
for i, (file_path, test_set) in enumerate(all_paths):
    for num_solutions_per_try in num_solutions_per_try_options:
        print(test_set, f'{i}/{len(all_paths)}', False,num_solutions_per_try, time.time()-t)
        target_id = file_path.split('\\')[-1].replace('.png', '')
        run_name = f'{target_id}_decoder_{num_solutions_per_try}_False'
        if run_name + ".json" in os.listdir(results_dir):
            continue

        mcts_operator.set_new_target(file_path, mcts_workdir, starting_point_btm=None)
        image_tensor = dataset.open_image(file_path).to(device)
        image_batch = image_tensor.unsqueeze(0).repeat(num_solutions_per_try, 1, 1, 1)
        pre_inference = time.time()
        generated_codes = loaded_model.generate(
            image_batch, 
            max_new_tokens=MAX_SEQUENCE_LENGTH, 
            greedy=False,
            temperature=0.8,
            top_p=0.9
        )
        inference_time = time.time() - pre_inference
        new_images_paths = []
        nms = []
        for j, code in enumerate(generated_codes):
            try:
                nm = BlenderTreeManager.from_str(code)
                new_img_path = os.path.join(iterations_dir, run_name, f'{j}.png')
                generate_image(nm, new_img_path)
                new_images_paths.append(new_img_path)
                nms.append(nm)
            except:  # sometime the generated code has errors and cannot compile
                pass
        if len(new_images_paths) == 0:
            continue
        scores = mcts_operator.compare_images_to_target(new_images_paths).flatten()
        
        best_score, best_path = max(zip(scores, new_images_paths))
        res = {
            'target_id': target_id,
            'target_path': file_path,
            'method': 'decoder',
            'optimization': False,
            'test_set': test_set,
            'best_similarity': best_score,
            'best_image_path': best_path,
            'inference_times': [inference_time],
            'num_iterations': num_solutions_per_try,
            'best_tree_text': '',
            'iterations_log': None,
        }
        with open(os.path.join(results_dir, run_name + ".json"), "w") as f:
            json.dump(res, f, indent=4, default=json_safe)
        clear_output()

new_textures 0/233 False 7 0.0
new_textures 0/233 False 2 0.0
new_textures 0/233 False 20 0.0
new_textures 1/233 False 7 0.008198022842407227
new_textures 1/233 False 2 0.008198022842407227
new_textures 1/233 False 20 0.008198022842407227
new_textures 2/233 False 7 0.008198022842407227
new_textures 2/233 False 2 0.008198022842407227
new_textures 2/233 False 20 0.008198022842407227
new_textures 3/233 False 7 0.008198022842407227
new_textures 3/233 False 2 0.008198022842407227
new_textures 3/233 False 20 0.008198022842407227
new_textures 4/233 False 7 0.008198022842407227
new_textures 4/233 False 2 0.008198022842407227
new_textures 4/233 False 20 0.008198022842407227
new_textures 5/233 False 7 0.008198022842407227
new_textures 5/233 False 2 0.008198022842407227
new_textures 5/233 False 20 0.008198022842407227
new_textures 6/233 False 7 0.008198022842407227
new_textures 6/233 False 2 0.02433490753173828
new_textures 6/233 False 20 0.02433490753173828
new_textures 7/233 False 7 0.024334907